In [ ]:
import sys
import os
# Add pipeline directory to sys.path to enable imports
sys.path.append(os.path.abspath('../pipeline'))


In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [ ]:
ANGLE_COLS = [
    "L_Shoulder_Angle", "R_Shoulder_Angle",
    "L_Elbow_Angle", "R_Elbow_Angle",
    "L_Hip_Angle", "R_Hip_Angle",
    "L_Knee_Angle", "R_Knee_Angle",
]

LANDMARK_KEYS = [
    "Head", "L_Sho", "R_Sho",
    "L_Elb", "R_Elb", "L_Wri", "R_Wri",
    "L_Hip", "R_Hip", "L_Kne", "R_Kne",
    "L_Ank", "R_Ank", "Mid_Hip",
]

FEATURE_COLS = []
for angle in ANGLE_COLS:
    FEATURE_COLS.extend(
        [f"{angle}_max", f"{angle}_min", f"{angle}_avg", f"{angle}_var"]
    )
for lm in LANDMARK_KEYS:
    FEATURE_COLS.extend([f"{lm}_x_std", f"{lm}_y_std"])


def summarize_angles(df_angles: pd.DataFrame) -> pd.DataFrame:
    df = df_angles[["Sequence_ID"] + ANGLE_COLS].copy()
    for col in ANGLE_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    grouped = df.groupby("Sequence_ID", as_index=True)
    summary_parts = []
    for col in ANGLE_COLS:
        g = grouped[col]
        summary_parts.append(
            pd.DataFrame(
                {
                    f"{col}_max": g.max(),
                    f"{col}_min": g.min(),
                    f"{col}_avg": g.mean(),
                    f"{col}_var": g.var(ddof=0),
                }
            )
        )
    return pd.concat(summary_parts, axis=1)


def summarize_joints(df_joints: pd.DataFrame) -> pd.DataFrame:
    df = df_joints.copy()

    if "Mid_Hip_x" not in df.columns and {"L_Hip_x", "R_Hip_x"}.issubset(df.columns):
        df["Mid_Hip_x"] = (
            pd.to_numeric(df["L_Hip_x"], errors="coerce")
            + pd.to_numeric(df["R_Hip_x"], errors="coerce")
        ) / 2
    if "Mid_Hip_y" not in df.columns and {"L_Hip_y", "R_Hip_y"}.issubset(df.columns):
        df["Mid_Hip_y"] = (
            pd.to_numeric(df["L_Hip_y"], errors="coerce")
            + pd.to_numeric(df["R_Hip_y"], errors="coerce")
        ) / 2

    for lm in LANDMARK_KEYS:
        for axis in ("x", "y"):
            col = f"{lm}_{axis}"
            if col not in df.columns:
                df[col] = np.nan
            df[col] = pd.to_numeric(df[col], errors="coerce")

    coord_cols = [f"{lm}_x" for lm in LANDMARK_KEYS] + [f"{lm}_y" for lm in LANDMARK_KEYS]
    summary = df.groupby("Sequence_ID", as_index=True)[coord_cols].std(ddof=0)
    summary = summary.rename(columns={c: f"{c}_std" for c in coord_cols})
    return summary


def load_action_features(label: str, angles_path: Path, joints_path: Path) -> pd.DataFrame:
    angles = pd.read_csv(angles_path)
    joints = pd.read_csv(joints_path)

    angle_summary = summarize_angles(angles)
    joint_summary = summarize_joints(joints)

    merged = angle_summary.join(joint_summary, how="inner")
    merged["action_label"] = label
    return merged.reset_index()


root = Path(".")
data_files = {
    "pushup": {
        "angles": root / "Pushup" / "pushup_angles.csv",
        "joints": root / "Pushup" / "pushup_joints.csv",
    },
    "squat": {
        "angles": root / "Squat" / "squat_angles.csv",
        "joints": root / "Squat" / "squat_joints.csv",
    },
    "jump_rope": {
        "angles": root / "Jump_Rope" / "jump_rope_angles.csv",
        "joints": root / "Jump_Rope" / "jump_rope_joints.csv",
    },
    "pullup": {
        "angles": root / "Pullup" / "pullup_angles.csv",
        "joints": root / "Pullup" / "pullup_joints.csv",
    },
}

frames = [
    load_action_features(label, files["angles"], files["joints"])
    for label, files in data_files.items()
]
df = pd.concat(frames, ignore_index=True)

for col in FEATURE_COLS:
    if col not in df.columns:
        df[col] = np.nan

df = df.reindex(columns=["Sequence_ID", "action_label"] + FEATURE_COLS)

feature_cols = FEATURE_COLS
X_raw = df[feature_cols]
y = df["action_label"]

imputer = SimpleImputer(strategy="constant", fill_value=0)
X_imputed = pd.DataFrame(imputer.fit_transform(X_raw), columns=feature_cols)

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=feature_cols)
joblib.dump(scaler, "scaler.pkl")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

action_clf = RandomForestClassifier(
    n_estimators=200, random_state=42, n_jobs=-1
)
action_clf.fit(X_train, y_train)

y_pred = action_clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Action classifier accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred))

joblib.dump(action_clf, "action_classifier.pkl")

Action classifier accuracy: 0.9880
              precision    recall  f1-score   support

   jump_rope       0.97      1.00      0.99        39
      pullup       0.98      1.00      0.99        40
      pushup       1.00      1.00      1.00        42
       squat       1.00      0.96      0.98        46

    accuracy                           0.99       167
   macro avg       0.99      0.99      0.99       167
weighted avg       0.99      0.99      0.99       167



['action_classifier.pkl']

In [ ]:
quality_scores = np.zeros(len(df), dtype=float)

for label in df["action_label"].unique():
    idx = df["action_label"] == label
    X_group = X_scaled.loc[idx]

    iso = IsolationForest(
        n_estimators=200,
        contamination="auto",
        random_state=42,
        n_jobs=-1,
    )
    iso.fit(X_group)

    scores = iso.decision_function(X_group).reshape(-1, 1)
    score_scaler = MinMaxScaler(feature_range=(0.4, 1.0))
    scaled_scores = score_scaler.fit_transform(scores).ravel()

    quality_scores[idx.to_numpy()] = scaled_scores

df["quality_score"] = quality_scores

In [ ]:
action_ohe = pd.get_dummies(df["action_label"], prefix="action")
expected_ohe_cols = [f"action_{a}" for a in sorted(data_files.keys())]
action_ohe = action_ohe.reindex(columns=expected_ohe_cols, fill_value=0)
action_ohe_cols = action_ohe.columns.tolist()

X_quality = pd.concat(
    [X_scaled.reset_index(drop=True), action_ohe.reset_index(drop=True)],
    axis=1,
)
regressor_feature_cols = X_quality.columns.tolist()
y_quality = df["quality_score"].values

Xq_train, Xq_test, yq_train, yq_test = train_test_split(
    X_quality, y_quality, test_size=0.2, random_state=42
)

quality_reg = RandomForestRegressor(
    n_estimators=300, random_state=42, n_jobs=-1
)
quality_reg.fit(Xq_train, yq_train)

yq_pred = quality_reg.predict(Xq_test)
mse = mean_squared_error(yq_test, yq_pred)
print(f"Quality regressor MSE: {mse:.6f}")

joblib.dump(quality_reg, "quality_regressor.pkl")

Quality regressor MSE: 0.006461


['quality_regressor.pkl']

In [ ]:
sample_idx = np.random.randint(0, len(df))
sample_row = df.iloc[sample_idx]

sample_features = pd.DataFrame([sample_row[feature_cols].values], columns=feature_cols)
sample_features_imputed = pd.DataFrame(
    imputer.transform(sample_features), columns=feature_cols
)
sample_features_scaled = pd.DataFrame(
    scaler.transform(sample_features_imputed), columns=feature_cols
)

pred_action = action_clf.predict(sample_features_scaled)[0]

sample_action_ohe = pd.DataFrame(
    np.zeros((1, len(action_ohe_cols))), columns=action_ohe_cols
)
action_col = f"action_{pred_action}"
if action_col in sample_action_ohe.columns:
    sample_action_ohe.loc[0, action_col] = 1

sample_quality_input = pd.concat([sample_features_scaled, sample_action_ohe], axis=1)
sample_quality_input = sample_quality_input.reindex(
    columns=regressor_feature_cols, fill_value=0
)

pred_quality = quality_reg.predict(sample_quality_input)[0]

print(f"Predicted action: {pred_action}")
print(f"Predicted form score: {pred_quality:.4f}")

Predicted action: jump_rope
Predicted form score: 0.9791
